In [1]:
import hail as hl
hl.init(spark_conf={'spark.driver.memory': '32g'})
vcf_norm = "/path/to/local_data/EGA_recall/output-joint/output-joint_norm_func.vcf.bgz"

SLF4J: No SLF4J providers were found.
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See https://www.slf4j.org/codes.html#noProviders for further details.
SLF4J: Class path contains SLF4J bindings targeting slf4j-api versions 1.7.x or earlier.
SLF4J: Ignoring binding found at [jar:file:/path/to/data/mambaforge/envs/hail/lib/python3.10/site-packages/pyspark/jars/log4j-slf4j-impl-2.17.2.jar!/org/slf4j/impl/StaticLoggerBinder.class]
SLF4J: See https://www.slf4j.org/codes.html#ignoredBindings for an explanation.
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
Running on Apache Spark version 3.3.2
SparkUI available at http://unknown04cf4b20a517.attlocal.net:4040
Welcome to
     __  __     <>__
    / /_/ /__  __/ /
   / __  / _ `/ / /
  /_/ /_/\_,_/_/_/   version 0.2.115-10932c754edb
LOGGING: writing to /path/to/data/projects/project/EGA_recalling/hail-20230501-1033-0.2.115-10932c754edb.lo

In [2]:
from hail.plot import show
from pprint import pprint
import pandas as pd
hl.plot.output_notebook()
rg_new = hl.ReferenceGenome.from_fasta_file(fasta_file='/path/to/local_data/references/human_g1k_v37_fixedChr.fasta',
                                            name="hg19",
                                            index_file='/path/to/local_data/references/human_g1k_v37_fixedChr.fasta.fai')

Loading BokehJS ...

In [3]:
hl.import_vcf(vcf_norm, reference_genome="hg19").write('/path/to/local_data/EGA_recall/output-joint/hail.mt', overwrite=True)
mt = hl.read_matrix_table('/path/to/local_data/EGA_recall/output-joint/hail.mt')
table = (hl.import_table('/path/to/local_data/EGA_recall/EGA_metadata.tsv', impute=True)
         .key_by('sample_original'))

2023-05-01 10:34:03.193 Hail: INFO: scanning VCF for sortedness...
2023-05-01 10:34:16.168 Hail: INFO: Coerced prefix-sorted VCF, requiring additional sorting within data partitions on each query.
2023-05-01 10:34:56.263 Hail: INFO: wrote matrix table with 658632 rows and 372 columns in 9 partitions to /path/to/local_data/EGA_recall/output-joint/hail.mt
2023-05-01 10:35:00.269 Hail: INFO: Reading table to impute column types
2023-05-01 10:35:02.680 Hail: INFO: Finished type imputation
  Loading field 'sample_original' as type str (imputed)
  Loading field 'sample_mod1' as type str (imputed)
  Loading field 'sample_mod2' as type str (imputed)
  Loading field 'clustering_notes' as type str (imputed)
  Loading field 'tumor_normal' as type str (imputed)
  Loading field 'patient' as type str (imputed)
  Loading field 'study' as type str (imputed)
  Loading field 'mean_dp' as type float64 (imputed)
  Loading field 'call_rate' as type float64 (imputed)


In [5]:
mt = mt.annotate_cols(pheno = table[mt.s])
mt = mt.key_cols_by(mt.pheno.sample_mod2)
mt.col.describe()
mt.describe()

--------------------------------------------------------
Type:
        struct {
        s: str, 
        pheno: struct {
            sample_mod1: str, 
            sample_mod2: str, 
            clustering_notes: str, 
            tumor_normal: str, 
            patient: str, 
            study: str, 
            mean_dp: float64, 
            call_rate: float64
        }, 
        sample_mod2: str
    }
--------------------------------------------------------
Source:
Index:
    ['column']
--------------------------------------------------------
----------------------------------------
Global fields:
    None
----------------------------------------
Column fields:
    's': str
    'pheno': struct {
        sample_mod1: str, 
        sample_mod2: str, 
        clustering_notes: str, 
        tumor_normal: str, 
        patient: str, 
        study: str, 
        mean_dp: float64, 
        call_rate: float64
    }
    'sample_mod2': str
----------------------------------------
Row fields

2023-05-01 10:36:16.509 Hail: WARN: cols(): Resulting column table is sorted by 'col_key'.
    To preserve matrix table column order, first unkey columns with 'key_cols_by()'


In [6]:
p = hl.plot.histogram(mt.DP, range=(0,200), bins=30, title='DP Histogram', legend='DP')
show(p)
mt = hl.sample_qc(mt)



In [7]:
p = hl.plot.histogram(mt.sample_qc.call_rate, range=(.75,1), legend='Call Rate')
show(p)

In [8]:
p = hl.plot.histogram(mt.sample_qc.gq_stats.mean, range=(10,85), legend='Mean Sample GQ')
show(p)



In [9]:
p = hl.plot.scatter(mt.sample_qc.dp_stats.mean, mt.sample_qc.call_rate, xlabel='Mean DP', ylabel='Call Rate')
show(p)


/path/to/data/mambaforge/envs/hail/lib/python3.10/site-packages/bokeh/models/sources.py:235: FutureWarning: iteritems is deprecated and will be removed in a future version. Use .items instead.
  tmp_data = {c: v.values for c, v in _df.iteritems()}


In [10]:
# filter columns and rows
a = pd.DataFrame({"mean_dp": mt.sample_qc.dp_stats.mean.collect(), "call_rate":mt.sample_qc.call_rate.collect()}, index=mt.cols().sample_mod2.collect())
a.to_csv("/path/to/local_data/EGA_recall/calling_stats.tsv", sep="\t")

mt = mt.filter_cols((mt.sample_qc.dp_stats.mean >= 10) & (mt.sample_qc.call_rate >= 0.8))
print('After filter, %d/372 samples remain.' % mt.count_cols())
mt.count()
ab = mt.AD[1] / hl.sum(mt.AD)
filter_condition_ab = ((mt.GT.is_hom_ref() & (ab <= 0.1)) |
                        (mt.GT.is_het() & (ab >= 0.25) & (ab <= 0.75)) |
                        (mt.GT.is_hom_var() & (ab >= 0.9)))

fraction_filtered = mt.aggregate_entries(hl.agg.fraction(~filter_condition_ab))
print(f'Filtering {fraction_filtered * 100:.2f}% entries out of downstream analysis.')
mt = mt.filter_entries(filter_condition_ab)

2023-05-01 10:39:01.946 Hail: INFO: Coerced sorted dataset===>      (8 + 1) / 9]
2023-05-01 10:39:02.138 Hail: INFO: Ordering unsorted dataset with network shuffle


After filter, 370/372 samples remain.


Filtering 6.01% entries out of downstream analysis.


In [11]:
mt = hl.variant_qc(mt)
mt.row.describe()

--------------------------------------------------------
Type:
        struct {
        locus: locus<hg19>, 
        alleles: array<str>, 
        rsid: str, 
        qual: float64, 
        filters: set<str>, 
        info: struct {
            AC: array<int32>, 
            AF: array<float64>, 
            AN: int32, 
            BaseQRankSum: float64, 
            ClippingRankSum: float64, 
            DB: bool, 
            DP: int32, 
            ExcessHet: float64, 
            FS: float64, 
            FUNCOTATION: array<str>, 
            InbreedingCoeff: float64, 
            MLEAC: array<int32>, 
            MLEAF: array<float64>, 
            MQ: float64, 
            MQRankSum: float64, 
            QD: float64, 
            ReadPosRankSum: float64, 
            SOR: float64
        }, 
        variant_qc: struct {
            dp_stats: struct {
                mean: float64, 
                stdev: float64, 
                min: float64, 
                max: float64
     

In [12]:
ibd = hl.identity_by_descent(mt) 

2023-05-01 10:39:19.911 Hail: INFO: Coerced sorted dataset
2023-05-01 10:39:20.245 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:39:24.852 Hail: INFO: Coerced sorted dataset          (7 + 2) / 9]
2023-05-01 10:39:25.015 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:40:12.804 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:40:14.687 Hail: INFO: wrote table with 68265 rows in 9 partitions to /tmp/persist_TableZrpTnkCF2q


In [13]:
ibd.show()

+-------------------+-------------------+----------+----------+----------+
| i                 | j                 |   ibd.Z0 |   ibd.Z1 |   ibd.Z2 |
+-------------------+-------------------+----------+----------+----------+
| str               | str               |  float64 |  float64 |  float64 |
+-------------------+-------------------+----------+----------+----------+
| "1711STDY5270146" | "1711STDY5270147" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "1711STDY5270148" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "1712STDY5304131" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "1712STDY5304137" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "1712STDY5304145" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "1712STDY5304153" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "1712STDY5304154" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "1712STDY5368618" | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "PD4060a"         | 1.00e+00 | 0.00e+00 | 0.00e+00 |
| "1711STDY5270146" | "PD4060b2"        | 7.37e-01 | 0.00e+00 | 2.63e-01 |
+-------------------+-------------------+----------+----------+----------+

+------------+-------+-------+--------+
| ibd.PI_HAT |  ibs0 |  ibs1 |   ibs2 |
+------------+-------+-------+--------+
|    float64 | int64 | int64 |  int64 |
+------------+-------+-------+--------+
|   0.00e+00 | 10006 | 59408 | 525101 |
|   0.00e+00 | 10202 | 60587 | 527015 |
|   0.00e+00 | 10292 | 58242 | 518331 |
|   0.00e+00 | 10446 | 58876 | 521331 |
|   0.00e+00 | 10011 | 58240 | 518569 |
|   0.00e+00 | 10200 | 58496 | 518660 |
|   0.00e+00 | 10250 | 58270 | 517346 |
|   0.00e+00 | 10031 | 54159 | 497769 |
|   0.00e+00 |  9596 | 58003 | 501008 |
|   2.63e-01 |  7927 | 58645 | 524785 |
+------------+-------+-------+--------+
showing top 10 rows

In [14]:
pc_rel = hl.pc_relate(mt.GT, 0.001, k=2, statistics='kin')

2023-05-01 10:40:25.721 Hail: INFO: hwe_normalize: found 581907 variants after filtering out monomorphic sites.
2023-05-01 10:40:27.217 Hail: INFO: Coerced sorted dataset
2023-05-01 10:40:27.643 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:40:32.847 Hail: INFO: Coerced sorted dataset          (7 + 2) / 9]
2023-05-01 10:40:32.992 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:40:38.448 Hail: INFO: pca: running PCA with 2 components...2) / 9]
2023-05-01 10:41:07.021 Hail: INFO: wrote table with 0 rows in 0 partitions to /tmp/persist_TableRNrakIjwYb
2023-05-01 10:41:08.400 Hail: INFO: Coerced sorted dataset
2023-05-01 10:41:08.852 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:41:13.593 Hail: INFO: Coerced sorted dataset===>      (8 + 1) / 9]
2023-05-01 10:41:13.731 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:41:16.393 Hail: INFO: Coerced sorted dataset
2023-05-01 10:41:16.551 Hail

In [15]:
pairs = pc_rel.filter(pc_rel['kin'] > 0.125)
related_samples_to_remove = hl.maximal_independent_set(pairs.i, pairs.j,
                                                       keep=False)

result = mt.filter_cols(
    hl.is_defined(related_samples_to_remove[mt.col_key]), keep=False)

result.write('/path/to/local_data/EGA_recall/output-joint/hail_filtered.mt', overwrite=True)

2023-05-01 10:45:24.203 Hail: INFO: wrote table with 191 rows in 1 partition to /tmp/OHAGf6Fe8Ca9KLzbvjr8H0
2023-05-01 10:45:25.794 Hail: INFO: Coerced sorted dataset
2023-05-01 10:45:25.914 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:45:37.698 Hail: INFO: Coerced sorted dataset===>      (8 + 1) / 9]
2023-05-01 10:45:37.891 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:45:42.311 Hail: INFO: Coerced sorted dataset          (5 + 4) / 9]
2023-05-01 10:45:42.433 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:45:43.951 Hail: INFO: Coerced sorted dataset
2023-05-01 10:45:44.320 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:45:46.326 Hail: INFO: Coerced sorted dataset
2023-05-01 10:45:47.035 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:46:12.689 Hail: INFO: wrote matrix table with 658632 rows and 190 columns in 9 partitions to /path/to/local_data/EGA_recall/

In [16]:
p = hl.plot.histogram(pc_rel['kin'], bins=30, title='pc_rel Histogram', legend='kin')
show(p)

In [17]:
result.count()

2023-05-01 10:46:20.234 Hail: INFO: Coerced sorted dataset          (5 + 4) / 9]
2023-05-01 10:46:20.423 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:46:25.236 Hail: INFO: Coerced sorted dataset          (7 + 2) / 9]
2023-05-01 10:46:25.482 Hail: INFO: Ordering unsorted dataset with network shuffle
2023-05-01 10:46:26.955 Hail: INFO: Coerced sorted dataset
2023-05-01 10:46:27.186 Hail: INFO: Ordering unsorted dataset with network shuffle


(658632, 190)